# 03 — Exploratory Data Analysis

## Saudi Household Electricity Consumption

This notebook performs a publication-quality exploratory analysis of the two **validated,
frozen Phase 1 panels**. It does not recreate integration, modify source values, or use
historical TechnicalWork outputs.

The analysis covers data quality, distributions, regional and temporal variation, seasonal
patterns, costs, correlations, descriptive outliers, and temporal consistency. Outlier flags
are analytical annotations only; observations are not removed.


## 1. Imports and reproducible plotting configuration


In [ ]:
from pathlib import Path
from datetime import datetime
from copy import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="colorblind",
    rc={"figure.dpi": 120, "savefig.dpi": 300},
)
COLORS = {
    "navy": "#1F3A5F",
    "teal": "#2A9D8F",
    "gold": "#E9C46A",
    "orange": "#F4A261",
    "red": "#C8553D",
    "grey": "#6B7280",
}


## 2. Locate the project and load only validated Phase 1 data


In [ ]:
def locate_project_root(start: Path) -> Path:
    '''Locate the repository without relying on an absolute machine path.'''
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Start Jupyter from the project root or notebooks directory."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

for directory in (RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

ANNUAL_PATH = PROCESSED_DIR / "saudi_household_admin_region_annual_panel.csv"
SEASONAL_PATH = PROCESSED_DIR / "saudi_household_admin_region_seasonal_panel.csv"

if not ANNUAL_PATH.exists() or not SEASONAL_PATH.exists():
    raise FileNotFoundError(
        "Validated Phase 1 panels are required. Run the frozen Phase 1 notebooks first."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Annual input: {ANNUAL_PATH.relative_to(PROJECT_ROOT)}")
print(f"Seasonal input: {SEASONAL_PATH.relative_to(PROJECT_ROOT)}")


In [ ]:
annual = pd.read_csv(ANNUAL_PATH)
seasonal = pd.read_csv(SEASONAL_PATH)

annual = annual.sort_values(["region", "year"]).reset_index(drop=True)
seasonal = seasonal.sort_values(["region", "year", "season"]).reset_index(drop=True)

print(f"Annual panel dimensions: {annual.shape}")
print(f"Seasonal panel dimensions: {seasonal.shape}")
display(annual.head())
display(seasonal.head())


## 3. Dataset overview and variable descriptions

The annual panel contains one observation per administrative region and year. The seasonal
panel contains two components per region-year: `Winter` and `Rest of the year`. Consumption
is measured in kWh and cost in SAR. Provenance columns retain the source selection and
corroboration decisions made in frozen Phase 1.


In [ ]:
variable_descriptions = pd.DataFrame([
    ("year", "Calendar/reporting year"),
    ("region", "Canonical Saudi administrative region"),
    ("annual_consumption_kwh", "Winter plus rest-of-year household electricity consumption (kWh)"),
    ("annual_cost_sar", "Winter plus rest-of-year household electricity cost (SAR)"),
    ("average_cost_sar_per_kwh", "Annual cost divided by annual consumption"),
    ("winter_consumption_share_pct", "Winter consumption as a percentage of annual consumption"),
    ("season", "Published seasonal component"),
    ("consumption_kwh", "Seasonal household electricity consumption (kWh)"),
    ("cost_sar", "Seasonal household electricity cost (SAR)"),
])
variable_descriptions.columns = ["variable", "description"]
display(variable_descriptions)

print("Annual data types:")
display(annual.dtypes.rename("dtype").to_frame())
print("Seasonal data types:")
display(seasonal.dtypes.rename("dtype").to_frame())


## 4. Remaining quality verification


In [ ]:
annual_key = ["year", "region"]
seasonal_key = ["year", "region", "season"]
required_annual = [
    "year", "region", "winter_consumption_kwh",
    "rest_of_year_consumption_kwh", "winter_cost_sar",
    "rest_of_year_cost_sar", "annual_consumption_kwh",
    "annual_cost_sar", "average_cost_sar_per_kwh",
    "winter_consumption_share_pct",
]
required_seasonal = ["year", "region", "season", "consumption_kwh", "cost_sar"]

quality_checks = {
    "annual_missing_required": int(annual[required_annual].isna().sum().sum()),
    "seasonal_missing_required": int(seasonal[required_seasonal].isna().sum().sum()),
    "annual_duplicate_keys": int(annual.duplicated(annual_key).sum()),
    "seasonal_duplicate_keys": int(seasonal.duplicated(seasonal_key).sum()),
    "nonpositive_annual_consumption": int((annual["annual_consumption_kwh"] <= 0).sum()),
    "nonpositive_annual_cost": int((annual["annual_cost_sar"] <= 0).sum()),
    "nonpositive_seasonal_consumption": int((seasonal["consumption_kwh"] <= 0).sum()),
    "nonpositive_seasonal_cost": int((seasonal["cost_sar"] <= 0).sum()),
}
quality_table = pd.Series(quality_checks, name="count").to_frame()
display(quality_table)

if any(quality_checks.values()):
    raise ValueError("A required quality check failed; inspect the displayed counts.")


## 5. Summary statistics and distribution analysis


In [ ]:
analysis_columns = [
    "annual_consumption_kwh", "annual_cost_sar",
    "average_cost_sar_per_kwh", "winter_consumption_share_pct",
]
summary_statistics = annual[analysis_columns].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
).T
display(summary_statistics)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
labels = {
    "annual_consumption_kwh": "Annual consumption (billion kWh)",
    "annual_cost_sar": "Annual cost (billion SAR)",
    "average_cost_sar_per_kwh": "Average cost (SAR/kWh)",
    "winter_consumption_share_pct": "Winter share (%)",
}
scales = {
    "annual_consumption_kwh": 1e9,
    "annual_cost_sar": 1e9,
    "average_cost_sar_per_kwh": 1,
    "winter_consumption_share_pct": 1,
}
for ax, column in zip(axes.flat, analysis_columns):
    values = annual[column] / scales[column]
    sns.histplot(values, kde=True, ax=ax, color=COLORS["teal"], edgecolor="white")
    ax.set_xlabel(labels[column])
    ax.set_ylabel("Observations")
    ax.set_title(labels[column])
fig.suptitle("Distributions of Validated Annual Household Variables", fontsize=17, weight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "variable_distributions.png", bbox_inches="tight")
plt.show()


## 6. Regional comparisons: consumption and cost


In [ ]:
region_summary = (
    annual.groupby("region", as_index=False)
    .agg(
        mean_consumption_kwh=("annual_consumption_kwh", "mean"),
        mean_cost_sar=("annual_cost_sar", "mean"),
    )
    .sort_values("mean_consumption_kwh")
)

fig, ax = plt.subplots(figsize=(11, 7))
sns.barplot(
    data=region_summary, y="region",
    x=region_summary["mean_consumption_kwh"] / 1e9,
    color=COLORS["navy"], ax=ax,
)
ax.set_title("Mean Annual Household Electricity Consumption by Region", weight="bold")
ax.set_xlabel("Mean annual consumption (billion kWh)")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "annual_consumption_by_region.png", bbox_inches="tight")
plt.show()

region_cost = region_summary.sort_values("mean_cost_sar")
fig, ax = plt.subplots(figsize=(11, 7))
sns.barplot(
    data=region_cost, y="region",
    x=region_cost["mean_cost_sar"] / 1e9,
    color=COLORS["gold"], ax=ax,
)
ax.set_title("Mean Annual Household Electricity Cost by Region", weight="bold")
ax.set_xlabel("Mean annual cost (billion SAR)")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "annual_cost_by_region.png", bbox_inches="tight")
plt.show()


## 7. Annual trends and time-series overview


In [ ]:
national_annual = annual.groupby("year", as_index=False).agg(
    consumption_kwh=("annual_consumption_kwh", "sum"),
    cost_sar=("annual_cost_sar", "sum"),
)

fig, ax = plt.subplots(figsize=(13, 8))
for region, group in annual.groupby("region"):
    ax.plot(group["year"], group["annual_consumption_kwh"] / 1e9,
            marker="o", linewidth=1.7, label=region)
ax.set_title("Regional Household Electricity Consumption Trends", weight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Annual consumption (billion kWh)")
ax.legend(title="Region", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "regional_trends.png", bbox_inches="tight")
plt.show()

fig, ax1 = plt.subplots(figsize=(11, 6))
ax1.plot(national_annual["year"], national_annual["consumption_kwh"] / 1e9,
         color=COLORS["navy"], marker="o", linewidth=2.5, label="Consumption")
ax1.set_ylabel("Regional sum: consumption (billion kWh)", color=COLORS["navy"])
ax1.set_xlabel("Year")
ax2 = ax1.twinx()
ax2.plot(national_annual["year"], national_annual["cost_sar"] / 1e9,
         color=COLORS["orange"], marker="s", linewidth=2.5, label="Cost")
ax2.set_ylabel("Regional sum: cost (billion SAR)", color=COLORS["orange"])
ax1.set_title("Validated Household Electricity Time-Series Overview", weight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "time_series_overview.png", bbox_inches="tight")
plt.show()


## 8. Seasonal patterns


In [ ]:
season_year = seasonal.groupby(["year", "season"], as_index=False).agg(
    consumption_kwh=("consumption_kwh", "sum"),
    cost_sar=("cost_sar", "sum"),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.lineplot(
    data=season_year, x="year", y=season_year["consumption_kwh"] / 1e9,
    hue="season", marker="o", linewidth=2.3, ax=axes[0],
)
axes[0].set_title("Seasonal Consumption")
axes[0].set_ylabel("Regional sum (billion kWh)")
axes[0].set_xlabel("Year")
sns.lineplot(
    data=season_year, x="year", y=season_year["cost_sar"] / 1e9,
    hue="season", marker="o", linewidth=2.3, ax=axes[1],
)
axes[1].set_title("Seasonal Cost")
axes[1].set_ylabel("Regional sum (billion SAR)")
axes[1].set_xlabel("Year")
fig.suptitle("Published Seasonal Household Electricity Patterns", fontsize=17, weight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "seasonal_patterns.png", bbox_inches="tight")
plt.show()


## 9. Correlation analysis


In [ ]:
correlation_columns = [
    "winter_consumption_kwh", "rest_of_year_consumption_kwh",
    "winter_cost_sar", "rest_of_year_cost_sar",
    "annual_consumption_kwh", "annual_cost_sar",
    "average_cost_sar_per_kwh", "winter_consumption_share_pct",
]
corr = annual[correlation_columns].corr(method="pearson")
short_labels = [
    "Winter consumption", "Rest consumption", "Winter cost", "Rest cost",
    "Annual consumption", "Annual cost", "Average cost/kWh", "Winter share",
]
corr.index = short_labels
corr.columns = short_labels

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="vlag", center=0,
    vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
)
ax.set_title("Pearson Correlations Among Validated Annual Variables", weight="bold", pad=15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "correlation_heatmap.png", bbox_inches="tight")
plt.show()


## 10. Descriptive outlier identification

The interquartile-range rule is used to **flag** unusually large or small values within each
variable. Regional electricity data are structurally heterogeneous, so a flag is not evidence
of error. No observation is deleted or winsorized.


In [ ]:
outlier_records = []
for column in analysis_columns:
    q1, q3 = annual[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = annual.loc[
        (annual[column] < lower) | (annual[column] > upper),
        ["year", "region", column],
    ].copy()
    flagged["variable"] = column
    flagged["lower_bound"] = lower
    flagged["upper_bound"] = upper
    flagged["flagged_value"] = flagged[column]
    outlier_records.append(
        flagged[["year", "region", "variable", "flagged_value", "lower_bound", "upper_bound"]]
    )
outlier_table = pd.concat(outlier_records, ignore_index=True)
display(outlier_table)

plot_data = annual[["year", "region", *analysis_columns]].melt(
    id_vars=["year", "region"], var_name="variable", value_name="value"
)
plot_data["scaled_value"] = plot_data["value"]
plot_data.loc[plot_data["variable"].eq("annual_consumption_kwh"), "scaled_value"] /= 1e9
plot_data.loc[plot_data["variable"].eq("annual_cost_sar"), "scaled_value"] /= 1e9
plot_data["variable"] = plot_data["variable"].map(labels)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, variable in zip(axes.flat, labels.values()):
    subset = plot_data[plot_data["variable"].eq(variable)]
    sns.boxplot(data=subset, x="scaled_value", color=COLORS["teal"], ax=ax)
    sns.stripplot(data=subset, x="scaled_value", color=COLORS["navy"],
                  alpha=0.45, size=3, ax=ax)
    ax.set_title(variable)
    ax.set_xlabel("")
fig.suptitle("Descriptive IQR Outlier Analysis", fontsize=17, weight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "outlier_analysis.png", bbox_inches="tight")
plt.show()


## 11. Temporal consistency verification


In [ ]:
expected_years = list(range(int(annual["year"].min()), int(annual["year"].max()) + 1))
temporal_checks = []
for region, group in annual.groupby("region"):
    observed = sorted(group["year"].astype(int).tolist())
    temporal_checks.append({
        "region": region,
        "observed_years": ", ".join(map(str, observed)),
        "complete_sequence": observed == expected_years,
        "year_count": len(observed),
    })
temporal_consistency = pd.DataFrame(temporal_checks)

season_counts = (
    seasonal.groupby(["year", "region"])["season"].nunique().rename("season_count").reset_index()
)
display(temporal_consistency)
print("Incomplete regional sequences:", int((~temporal_consistency["complete_sequence"]).sum()))
print("Region-years without exactly two seasons:", int((season_counts["season_count"] != 2).sum()))

if not temporal_consistency["complete_sequence"].all():
    raise ValueError("At least one region has a discontinuous annual sequence.")
if not season_counts["season_count"].eq(2).all():
    raise ValueError("At least one region-year does not contain both seasonal components.")


## 12. Thesis-ready findings and reproducible summary


In [ ]:
top_consumption = region_summary.iloc[-1]
low_consumption = region_summary.iloc[0]
first_total = national_annual.iloc[0]["consumption_kwh"]
last_total = national_annual.iloc[-1]["consumption_kwh"]
overall_change_pct = (last_total / first_total - 1) * 100
median_price = annual["average_cost_sar_per_kwh"].median()
median_winter_share = annual["winter_consumption_share_pct"].median()

findings = [
    f"The validated annual panel contains {len(annual):,} observations for "
    f"{annual['region'].nunique()} regions over {annual['year'].min()}–{annual['year'].max()}.",
    f"The seasonal panel contains {len(seasonal):,} observations and exactly two "
    "published seasonal components for every region-year.",
    "No missing required analytical values, duplicate keys, non-positive consumption, "
    "or non-positive cost values remain.",
    f"{top_consumption['region']} has the highest mean annual household consumption "
    f"({top_consumption['mean_consumption_kwh']/1e9:,.2f} billion kWh), while "
    f"{low_consumption['region']} has the lowest "
    f"({low_consumption['mean_consumption_kwh']/1e9:,.2f} billion kWh).",
    f"The regional-sum annual consumption changed by {overall_change_pct:,.2f}% "
    f"between {int(national_annual.iloc[0]['year'])} and {int(national_annual.iloc[-1]['year'])}.",
    f"The median observed average cost is {median_price:,.4f} SAR/kWh, and the median "
    f"winter share is {median_winter_share:,.2f}%.",
    f"The IQR procedure flags {len(outlier_table)} variable-observations for review; "
    "these are retained because cross-region scale differences can be substantive.",
    "The panel is temporally balanced, but six annual observations per region are a "
    "short time series; later model evaluation must use chronological validation and "
    "report uncertainty cautiously.",
]

summary_lines = [
    "# Exploratory Analysis Summary",
    "",
    f"Generated: {datetime.now().astimezone().isoformat(timespec='seconds')}",
    "",
    "## Scope",
    "",
    "This analysis uses only the frozen Phase 1 validated annual and seasonal panels.",
    "",
    "## Data-quality verification",
    "",
]
summary_lines.extend([f"- **{name}**: {value}" for name, value in quality_checks.items()])
summary_lines.extend(["", "## Key findings", ""])
summary_lines.extend([f"- {finding}" for finding in findings])
summary_lines.extend([
    "",
    "## Interpretation limits",
    "",
    "- Associations and correlations are descriptive and do not establish causality.",
    "- IQR flags identify unusual scale, not necessarily erroneous observations.",
    "- Regional sums should not be described as independently published national totals.",
    "- The 2017–2018 source-provenance limitation documented in Phase 1 remains applicable.",
])
(RESULTS_DIR / "exploratory_summary.md").write_text(
    "\n".join(summary_lines) + "\n", encoding="utf-8"
)

print("\n".join(f"• {finding}" for finding in findings))
print("\nSaved: results/exploratory_summary.md and eight requested figures.")


## Conclusion

The validated data are complete, balanced, and internally consistent for exploratory analysis.
Strong differences in regional scale and meaningful seasonal structure justify region-aware,
chronologically ordered forecasting features. Descriptive outliers are retained. The short
2017–2022 history remains the principal modeling limitation and must shape later validation.
